In [1]:
# %% [code]
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

!pip install -q "transformers<5" datasets tqdm

import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from datasets import Dataset, config as datasets_config
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
    accuracy_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# disable TF inside datasets
import sys
datasets_config.TF_AVAILABLE = False
sys.modules.pop("tensorflow", None)

# ----------------- Load data -----------------
data_path = os.getenv("TOXIC_CORPUS_PATH", "../data/all_combined.csv")  # change to your path
df = pd.read_csv(data_path)
df["label"] = df["label"].astype(int)

print(df.shape)
df.head()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 40.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.
(84954, 4)


,text,label,source,script
0,Jo ata hy nea treqa batata hy kia lul mara how...,1,rut,roman
1,Lakh lanat ho terie shakal par... Aik chez dik...,1,rut,roman
2,Bakwaas bevqoof bna rahay ho logon ko,1,rut,roman
3,O lanti das bhi de,1,rut,roman
4,lun b gold ka hona chahiy tha,1,rut,roman


# Split + tokenize + loaders

In [2]:
# %% [code]
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42,
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok   = val_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in cols_to_keep])
val_tok   = val_tok.remove_columns([c for c in val_tok.column_names if c not in cols_to_keep])
test_tok  = test_tok.remove_columns([c for c in test_tok.column_names if c not in cols_to_keep])

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

batch_size = 32  # GPU can handle a bit more
train_loader = DataLoader(train_tok, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_tok, batch_size=64)
test_loader  = DataLoader(test_tok, batch_size=64)


Train: (67963, 4) Val: (8495, 4) Test: (8496, 4)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/67963 [00:00<?, ? examples/s]

Map:   0%|          | 0/8495 [00:00<?, ? examples/s]

Map:   0%|          | 0/8496 [00:00<?, ? examples/s]

# Model, optimizer, scheduler

In [5]:
# %% [code]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
model.to(device)

num_epochs = 4
optimizer = AdamW(model.parameters(), lr=2e-5)

num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


# Eval + training loop

In [6]:
# %% [code]
def eval_loop(data_loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            labels = batch["label"].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}

            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    avg_loss = total_loss / len(data_loader)
    return {"loss": avg_loss, "accuracy": acc, "precision": precision, "recall": recall, "f1": f1}, all_labels, all_preds


for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for i, batch in enumerate(pbar, start=1):
        labels = batch["label"].to(device)
        inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_train_loss += loss.item()
        pbar.set_postfix({"train_loss": total_train_loss / i})

    train_loss = total_train_loss / len(train_loader)
    val_metrics, _, _ = eval_loop(val_loader)
    print(f"\nEpoch {epoch+1}:")
    print(f"  Train loss: {train_loss:.4f}")
    print(
        f"  Val loss: {val_metrics['loss']:.4f} | "
        f"Acc: {val_metrics['accuracy']:.3f} | "
        f"Prec: {val_metrics['precision']:.3f} | "
        f"Rec: {val_metrics['recall']:.3f} | "
        f"F1: {val_metrics['f1']:.3f}"
    )


Epoch 1/4:   0%|          | 0/2124 [00:00<?, ?it/s]


Epoch 1:
  Train loss: 0.3106
  Val loss: 0.2374 | Acc: 0.896 | Prec: 0.843 | Rec: 0.671 | F1: 0.747


Epoch 2/4:   0%|          | 0/2124 [00:00<?, ?it/s]


Epoch 2:
  Train loss: 0.1995
  Val loss: 0.2014 | Acc: 0.910 | Prec: 0.798 | Rec: 0.816 | F1: 0.807


Epoch 3/4:   0%|          | 0/2124 [00:00<?, ?it/s]


Epoch 3:
  Train loss: 0.1470
  Val loss: 0.1961 | Acc: 0.918 | Prec: 0.828 | Rec: 0.812 | F1: 0.820


Epoch 4/4:   0%|          | 0/2124 [00:00<?, ?it/s]


Epoch 4:
  Train loss: 0.1085
  Val loss: 0.2214 | Acc: 0.921 | Prec: 0.836 | Rec: 0.818 | F1: 0.827


# Test metrics + save model

In [7]:
# %% [code]
test_metrics, y_true, y_pred = eval_loop(test_loader)

print("Test metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification report (test set):\n")
print(classification_report(y_true, y_pred, digits=3))

save_dir = Path(os.getenv("TOXIC_MODEL_OUTPUT_DIR", "../models/mbert_toxic_final"))
save_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model to:", save_dir)


Test metrics:
  loss: 0.2165
  accuracy: 0.9184
  precision: 0.8318
  recall: 0.8093
  f1: 0.8204

Classification report (test set):

              precision    recall  f1-score   support

           0      0.943     0.951     0.947      6540
           1      0.832     0.809     0.820      1956

    accuracy                          0.918      8496
   macro avg      0.888     0.880     0.884      8496
weighted avg      0.918     0.918     0.918      8496

Saved model to: /kaggle/working/mbert_toxic_final
